## 391_SAT_ID_ref_stockholmarchipelagotrail
* [#391](https://github.com/salgo60/Stockholm_Archipelago_Trail/issues/391)
* denna Notebook [391_SAT_ID_ref_stockholmarchipelagotrail.ipynb](https://github.com/salgo60/Stockholm_Archipelago_Trail/tree/main/Notebook/391_SAT_ID_ref_stockholmarchipelagotrail.ipynb)
* [ref:stockholmarchipelagotrail](https://wiki.openstreetmap.org/wiki/Sv:Key:ref:stockholmarchipelagotrail)

### Statistik  


| Tidpunkt | SAT OSM | OSM `unknown` | OSM med värde |
|----------|----:|--------------:|--------------:|
| 2026-06-13 13:31 | 326 | 126 | 436 |
| 2026-06-13 17:08 | 326 | 139 | 437 |
| 2026-06-15 22:27 | 338 | 216 | 656 |
| 2026-06-18 09:59 | 338 | 217 | 676 |
| 2026-06-24 05:26 | 380 | 207 | 697 |
| 2026-06-26 09:18 | 391 | 209 | 725 |
| 2026-06-26 19:51 | 470 | 194 | 732 |

In [1]:
import time
import datetime  
start_time = time.time()
start_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"Started: {start_str}")


Started: 2026-06-26 22:17


### antal objekt OSM 
* antal unknow

In [2]:

import requests

OVERPASS_URL = "https://overpass-api.de/api/interpreter"

query = """
[out:json][timeout:120];

nwr["ref:stockholmarchipelagotrail"="unknown"];
out count;

nwr["ref:stockholmarchipelagotrail"]["ref:stockholmarchipelagotrail"!="unknown"];
out count;

nwr["ref:stockholmarchipelagotrail"];
out count;
"""

headers = {
    "User-Agent": "MagnusOSMTools/1.0 (https://github.com/salgo60 salgo60@msn.com)"
}

response = requests.post(
    OVERPASS_URL,
    data={"data": query},
    headers=headers,
    timeout=180,
)

response.raise_for_status()

results = response.json()["elements"]

OSM_unknown = int(results[0]["tags"]["total"])
OSM_with_SAT = int(results[1]["tags"]["total"])
OSM_total = int(results[2]["tags"]["total"])

print(f"Unknown:        {OSM_unknown}")
print(f"Known with SAT: {OSM_with_SAT}")
print(f"Total with SAT: {OSM_total}")

Unknown:        192
Known with SAT: 540
Total with SAT: 732


### Antal objekt SAT geojson fil 

In [3]:

# Läs GeoJSON
url = "https://map.stockholmarchipelagotrail.com/data/geojson/pois.geojson"
geojson = requests.get(url).json()

checksSAT = []

for feature in geojson["features"]:
    props = feature["properties"]

    sat_id = props["id"]
    # Hämta tidsstämplar
    updated_at = props.get("updatedAt")
    first_seen = props.get("firstSeen")
    last_seen = props.get("lastSeen")   # om fältet finns


    osm_ref = None
    for item in props.get("sameAs", []):
        if item.startswith("osm:"):
            osm_ref = item
            break

    if osm_ref is None:
        continue

    _, osm_type, osm_id = osm_ref.split(":")

    checksSAT.append({
        "type": osm_type,
        "id": osm_id,
        "expected_ref": sat_id,
        "updatedAt": updated_at,
        "lastSeen": last_seen,
        "firstSeen": first_seen,

    })

all_SAT_nodes = len(checksSAT)
print(f"Antal SAT objekt att kontrollera: {all_SAT_nodes}")

Antal SAT objekt att kontrollera: 470


In [4]:
checksSAT.sort(
    key=lambda x: x["updatedAt"] or "",
    reverse=True
) 
print(sorted(geojson["features"][0]["properties"].keys()))

['address', 'category', 'description', 'fieldUpdatedAt', 'firstSeen', 'id', 'image', 'internet_access', 'name', 'nameLocalized', 'osmChangeset', 'osmVersion', 'passBuyUrl', 'passSells', 'passStamps', 'phone', 'reliability', 'sameAs', 'section', 'source', 'updatedAt', 'website', 'wikidata']


In [5]:
from urllib.parse import quote
import pandas as pd
from IPython.display import HTML, display

df = pd.DataFrame(checksSAT)

# Senaste först
df = df.sort_values("updatedAt", ascending=False)

# Klickbar länk till kartan
df["expected_ref"] = df["expected_ref"].apply(
    lambda x: (
        f'<a href="https://map.stockholmarchipelagotrail.com/?{x}" '
        f'target="_blank">{x}</a>'
    )
)

# Ny kolumn: JSON-LD/API
df["jsonld"] = df["expected_ref"].str.extract(r'>([^<]+)<')[0].apply(
    lambda x: (
        f'<a href="https://map.stockholmarchipelagotrail.com/api/objects/{quote(x, safe="")}" '
        f'target="_blank">jsonld</a>'
    )
)

display(
    HTML(
        df[
            [
                "updatedAt",
                "lastSeen",
                "firstSeen",
                "expected_ref",
                "jsonld",
                "type",
                "id",
            ]
        ].to_html(escape=False, index=False)
    )
)

updatedAt,lastSeen,firstSeen,expected_ref,jsonld,type,id
2026-06-26,None,2026-06-25,sat:poi:fgfa4,jsonld,node,13961732211
2026-06-26,None,2026-06-26,sat:poi:xswmg,jsonld,way,506814567
2026-06-26,None,2026-06-10,sat:poi:xczb8,jsonld,way,229110254
2026-06-26,None,2026-06-26,sat:poi:xh547,jsonld,way,833557518
2026-06-26,None,2026-06-10,sat:poi:r258p,jsonld,way,832213123
2026-06-26,None,2026-06-25,sat:poi:v57nn,jsonld,way,513323289
2026-06-26,None,2026-06-26,sat:poi:2cs5n,jsonld,way,1376435118
2026-06-26,None,2026-06-26,sat:poi:njv5p,jsonld,way,506100183
2026-06-26,None,2026-06-25,sat:poi:ej4zw,jsonld,way,388325539
2026-06-26,None,2026-06-26,sat:poi:5pwx4,jsonld,way,322935681


In [6]:
from urllib.parse import quote

df2 = pd.DataFrame(checksSAT)

# Skapa jsonld-kolumnen
df2["jsonld"] = df2["expected_ref"].apply(
    lambda x: f"https://map.stockholmarchipelagotrail.com/api/objects/{quote(x, safe='')}"
)

# Sortera efter firstSeen
df2 = df2.sort_values("firstSeen", ascending=False)

display(df2[
    [
        "firstSeen",
        "updatedAt",
        "lastSeen",
        "expected_ref",
        "jsonld",
        "type",
        "id",
    ]
])

,firstSeen,updatedAt,lastSeen,expected_ref,jsonld,type,id
437,2026-06-26,2025-05-12,None,sat:poi:6v7wh,https://map.stockholmarchipelagotrail.com/api/...,way,465662336
427,2026-06-26,2025-07-13,None,sat:poi:455vg,https://map.stockholmarchipelagotrail.com/api/...,way,857838617
417,2026-06-26,2026-06-07,None,sat:poi:yq34f,https://map.stockholmarchipelagotrail.com/api/...,way,322935629
418,2026-06-26,2026-05-15,None,sat:poi:3k92w,https://map.stockholmarchipelagotrail.com/api/...,way,330032369
419,2026-06-26,2025-09-17,None,sat:poi:eswrg,https://map.stockholmarchipelagotrail.com/api/...,node,1414841720
...,...,...,...,...,...,...,...
228,2026-06-10,2026-06-13,None,sat:poi:h9rbs,https://map.stockholmarchipelagotrail.com/api/...,node,12070083980
227,2026-06-10,2026-06-13,None,sat:poi:t7fj6,https://map.stockholmarchipelagotrail.com/api/...,node,11092271865
226,2026-06-10,2026-06-13,None,sat:poi:2ctyz,https://map.stockholmarchipelagotrail.com/api/...,node,11084999217
225,2026-06-10,2026-06-13,None,sat:poi:f4mjz,https://map.stockholmarchipelagotrail.com/api/...,node,11080859473


In [7]:
from urllib.parse import quote
from IPython.display import HTML

df = pd.DataFrame(checksSAT)

df["expected_ref"] = df["expected_ref"].apply(
    lambda x: f'<a href="https://map.stockholmarchipelagotrail.com/?{x}" target="_blank">{x}</a>'
)

df["jsonld"] = df["expected_ref"].str.extract(r'>([^<]+)<')[0].apply(
    lambda x: f'<a href="https://map.stockholmarchipelagotrail.com/api/objects/{quote(x, safe="")}" target="_blank">jsonld</a>'
)

df = df.sort_values("firstSeen", ascending=False)

HTML(df.to_html(escape=False, index=False))

type,id,expected_ref,updatedAt,lastSeen,firstSeen,jsonld
way,465662336,sat:poi:6v7wh,2025-05-12,None,2026-06-26,jsonld
way,857838617,sat:poi:455vg,2025-07-13,None,2026-06-26,jsonld
way,322935629,sat:poi:yq34f,2026-06-07,None,2026-06-26,jsonld
way,330032369,sat:poi:3k92w,2026-05-15,None,2026-06-26,jsonld
node,1414841720,sat:poi:eswrg,2025-09-17,None,2026-06-26,jsonld
way,836707934,sat:poi:x5weh,2025-09-01,None,2026-06-26,jsonld
way,698721756,sat:poi:2pscn,2025-08-29,None,2026-06-26,jsonld
way,1425797491,sat:poi:a9g3q,2025-08-29,None,2026-06-26,jsonld
way,1305399449,sat:poi:3t5da,2025-08-17,None,2026-06-26,jsonld
node,13057537355,sat:poi:n6qbm,2025-08-08,None,2026-06-26,jsonld


In [8]:
import pandas as pd

df = pd.DataFrame(checksSAT)
df = df.sort_values("updatedAt", ascending=False)

display(df)



,type,id,expected_ref,updatedAt,lastSeen,firstSeen
0,node,13961732211,sat:poi:fgfa4,2026-06-26,None,2026-06-25
6,way,506814567,sat:poi:xswmg,2026-06-26,None,2026-06-26
1,way,229110254,sat:poi:xczb8,2026-06-26,None,2026-06-10
9,way,833557518,sat:poi:xh547,2026-06-26,None,2026-06-26
8,way,832213123,sat:poi:r258p,2026-06-26,None,2026-06-10
...,...,...,...,...,...,...
465,way,223282382,sat:poi:267fb,2013-05-27,None,2026-06-26
466,way,223282393,sat:poi:cwaqz,2013-05-27,None,2026-06-26
467,node,1414841724,sat:poi:kaq8k,2012-11-13,None,2026-06-26
468,node,1414841723,sat:poi:7tqqb,2011-08-27,None,2026-06-26


## Resultat sista körningen 
* antal noder [SAT geojson](https://map.stockholmarchipelagotrail.com/data/geojson/pois.geojson)
* OSM ref_stockholmarchipelagotrail = unknown
  * noder med ref_stockholmarchipelagotrail som ej finns kopplade till SAT = unknown  
* antal noder i OSM med [ref_stockholmarchipelagotrail](https://overpass-turbo.eu/s/2rzp) / [karta](https://overpass-turbo.eu/s/2rzr)

Se även tags

In [9]:
print(f"| {start_str} | {all_SAT_nodes} | {OSM_unknown} | {OSM_total} |")  
#print(f"| {start_str} | {all_SAT_nodes} | {OSM_unknown_nodes} | {all_OSM_nodes} |")

| 2026-06-26 22:17 | 470 | 192 | 732 |


* jämför objekt i SAT geojson med OSM

In [10]:
from IPython.display import HTML, display

osm_lookup = {e["expected_ref"]: e for e in checksSAT}

rows = []

for feature in geojson["features"]:
    props = feature["properties"]
    sat_id = props["id"]

    if sat_id in osm_lookup:
        status = "✅ ok"
        osm = osm_lookup[sat_id]

        osm_link = (
            f"<a href='https://www.openstreetmap.org/{osm['type']}/{osm['id']}' "
            f"target='_blank'>{osm['type']}/{osm['id']}</a>"
        )

        unknown = (
            "⚠️ unknown"
            if osm.get("osm_ref") == "unknown"
            else ""
        )
    else:
        status = "❌ false"
        osm_link = ""
        unknown = ""

    rows.append(
        f"""
        <tr>
            <td>{status}</td>
            <td>{unknown}</td>
            <td>{sat_id}</td>
            <td>{props.get('name', '')}</td>
            <td>{osm_link}</td>
        </tr>
        """
    )

html = f"""
<table border="1">
    <tr>
        <th>Status</th>
        <th>OSM ref=unknown</th>
        <th>SAT ID</th>
        <th>Namn</th>
        <th>OSM</th>
    </tr>
    {''.join(rows)}
</table>
"""

display(HTML(html))

Status,OSM ref=unknown,SAT ID,Namn,OSM
✅ ok,,sat:poi:y2cyg,STF Finnhamns Vandrarhem,node/268908517
✅ ok,,sat:poi:n6mfb,Finnhamm Lanthandel,node/268910308
✅ ok,,sat:poi:w3rmg,Finnhamns café-krog,node/268910309
✅ ok,,sat:poi:sjwtd,Hamnboden,node/424073481
✅ ok,,sat:poi:z3q4b,Toalett,node/426925731
✅ ok,,sat:poi:9rwwe,Utö bykrog,node/435190209
✅ ok,,sat:poi:ftdh5,Svartsö Lanthandel,node/436368336
✅ ok,,sat:poi:jwkmw,Hamncaféet,node/443132453
✅ ok,,sat:poi:5a9dh,Hotell Furusund,node/443132454
✅ ok,,sat:poi:zt9z4,Furusundsmacken,node/443132455


### Sök ut wd och kolla om koppling osm finns
* kolla om OSM har ref:stockholmarchipelagotrail
* [SPARQL](https://w.wiki/R9Q2)


In [11]:
# saknar OSM 
import requests
import pandas as pd

query = """
SELECT DISTINCT
  ?id
  ?idLabel
  ?geo
WHERE {
  ?id wdt:P6104 wd:Q134294510 ;
      wdt:P625 ?geo .

  FILTER NOT EXISTS { ?id wdt:P11693 ?osmNode . }
  FILTER NOT EXISTS { ?id wdt:P10689 ?osmWay . }
  FILTER NOT EXISTS { ?id wdt:P402 ?osmRelation . }

  OPTIONAL { ?id wdt:P18 ?img . }

  SERVICE wikibase:label {
    bd:serviceParam wikibase:language "sv,en,de,en" .
  }
}
"""

url = "https://query.wikidata.org/sparql"
headers = {
    "Accept": "application/sparql-results+json",
    "User-Agent": "MagnusOSMTools/1.0 salgo60@msn.com"
}

response = requests.get(
    url,
    params={"query": query},
    headers=headers
)
response.raise_for_status()

data = response.json()["results"]["bindings"]

rows = []
for row in data:
    rows.append({
        "QID": row["id"]["value"].split("/")[-1],
        "Wikidata": row["id"]["value"],
        "idLabel": row.get("idLabel", {}).get("value", ""),
        "Koordinater": row.get("geo", {}).get("value", ""),
    })

df = pd.DataFrame(rows)

df["Wikidata"] = df["QID"].apply(
    lambda q: f'<a href="https://www.wikidata.org/wiki/{q}" target="_blank">{q}</a>'
)

from IPython.display import HTML
print("Saknar OSM objekt")
HTML(df.to_html(escape=False, index=False))

Saknar OSM objekt


QID,Wikidata,idLabel,Koordinater
Q115371401,Q115371401,Arholma kyrkogård,Point(19.110433 59.846011)
Q32301589,Q32301589,Nåttaröhals,Point(18.104166666 58.858333333)
Q136756996,Q136756996,Gamla skolhuset,Point(18.321663 58.969196)
Q134580717,Q134580717,Solberga gård Runmarö kajak uthyrning,Point(18.758125 59.255867)
Q1480015,Q1480015,Landsort,Point(17.8658 58.739603)
Q31895480,Q31895480,Edesnäs,Point(18.280694 58.954111)
Q133884947,Q133884947,Brottö kulturreservat,Point(18.727369 59.465922)
Q33108886,Q33108886,Yxlö,Point(18.838957892 59.607897461)
Q30315439,Q30315439,Sandhamns tullhus,Point(18.9147 59.28964)
Q135087816,Q135087816,Runmarö rum,Point(18.738225 59.277144)


In [12]:
import requests
import time
from tqdm.notebook import tqdm

OVERPASS_URL = "https://overpass-api.de/api/interpreter"

headers = {
    "User-Agent": "MagnusOSMTools/1.0 (salgo60@msn.com)"
}

osm_ref = []
osm_link = []

for qid in tqdm(df["QID"], total=len(df), desc="Kontrollerar OSM"):
    query = f"""
    [out:json][timeout:250];
    nwr["ref:stockholmarchipelagotrail"="{qid}"];
    out ids;
    """

    try:
        r = requests.post(OVERPASS_URL, data=query, headers=headers)
        r.raise_for_status()
        elements = r.json().get("elements", [])

        if elements:
            e = elements[0]
            osm_type = e["type"]  # node, way eller relation
            osm_id = e["id"]

            osm_ref.append("Ja")
            osm_link.append(
                f'<a href="https://www.openstreetmap.org/{osm_type}/{osm_id}" '
                f'target="_blank">{osm_type}/{osm_id}</a>'
            )
        else:
            osm_ref.append("Nej")
            osm_link.append("")

    except Exception as ex:
        osm_ref.append(f"Fel: {ex}")
        osm_link.append("")

    # Var snäll mot Overpass-servern
    time.sleep(40)

df["OSM_ref"] = osm_ref
df["OSM_länk"] = osm_link

from IPython.display import HTML
HTML(df.to_html(escape=False, index=False))

Kontrollerar OSM:   0%|          | 0/52 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
end_time = time.time()
duration = end_time - start_time
print(f"Started: {start_str}") 
print(f"Finished in {duration:.2f} seconds.")
